# Bind Mounts & tmpfs

Notebook 10 covered volumes — Docker-managed persistent storage. This notebook covers the other two storage types: **bind mounts** (mount a specific host path into a container) and **tmpfs** (mount RAM as a temporary filesystem inside the container).

Topics:
- Bind mounts: mounting host directories and files
- The development workflow: live code reloading
- Read-only bind mounts for configuration injection
- Bind mount pitfalls: UID/GID mismatches, macOS performance
- tmpfs: in-memory filesystems for sensitive or temporary data
- Choosing between volumes, bind mounts, and tmpfs

## 1. Bind Mounts

A bind mount maps a **specific path on the host** into the container. Unlike volumes, the host path is fully under your control — Docker doesn't manage it.

```bash
docker run -v /host/absolute/path:/container/path image
# or (explicit form)
docker run --mount type=bind,source=/host/path,target=/container/path image
```

**Key difference from volumes:** The host path must exist before `docker run`. If it doesn't exist, Docker (with `--mount`) throws an error. With `-v`, Docker creates the path as a directory on the host — which can be confusing.

### What you can bind-mount

| Source | Example |
|--------|--------|
| Directory | `-v $(pwd)/src:/app/src` |
| Single file | `-v $(pwd)/config.yaml:/app/config.yaml` |
| Socket | `-v /var/run/docker.sock:/var/run/docker.sock` |

Mounting a **single file** is useful for injecting configuration without copying it into the image.

In [ ]:
import os, textwrap

# Create a host directory with a file
os.makedirs("/tmp/bind-demo/src", exist_ok=True)

with open("/tmp/bind-demo/src/hello.py", "w") as f:
    f.write("print('Hello from bind-mounted source!')\n")

# The container sees the host file at /app/src/hello.py
!docker run --rm \
    --mount type=bind,source=/tmp/bind-demo/src,target=/app/src \
    python:3.12-slim \
    python /app/src/hello.py

In [ ]:
# Changes on the host are immediately visible inside the container
with open("/tmp/bind-demo/src/hello.py", "w") as f:
    f.write("print('Updated on the host — instantly visible in the container!')\n")

!docker run --rm \
    --mount type=bind,source=/tmp/bind-demo/src,target=/app/src \
    python:3.12-slim \
    python /app/src/hello.py

In [ ]:
# The container can also write back to the host
!docker run --rm \
    --mount type=bind,source=/tmp/bind-demo/src,target=/app/src \
    alpine sh -c 'echo "written by container" > /app/src/output.txt'

# Visible on the host
!cat /tmp/bind-demo/src/output.txt

## 2. The Development Workflow

Bind mounts are the cornerstone of the Docker development workflow. You mount your source code into the container so that code changes on the host are immediately reflected inside the running container — no rebuild required.

```bash
# Run a Flask app with hot-reload, source code mounted from host
docker run -d \
    --name flask-dev \
    -v $(pwd):/app \
    -p 5000:5000 \
    -e FLASK_ENV=development \
    myapp:dev
# Edit app.py on your laptop → Flask auto-reloads inside the container
```

This pattern means:
- Your editor, linter, and git remain on the host
- The runtime (Python, Node, JVM) stays in the container — no local installation
- Team members use the same runtime version regardless of their OS
- The production image is built with `COPY` — the source is baked in, no bind mount

In [ ]:
# Demonstrate dev workflow: mount src, run with auto-reload
os.makedirs("/tmp/devapp", exist_ok=True)

with open("/tmp/devapp/app.py", "w") as f:
    f.write(textwrap.dedent("""\
        from flask import Flask
        app = Flask(__name__)

        @app.route("/")
        def index():
            return "Version 1\\n"

        if __name__ == '__main__':
            app.run(host='0.0.0.0', port=5000, debug=True)
    """))

# Start dev server with source bind-mounted
!docker run -d --name flask-dev \
    -v /tmp/devapp:/app \
    -p 5001:5000 \
    python:3.12-slim \
    sh -c 'pip install flask -q && python /app/app.py'

import time; time.sleep(4)
!curl -s http://localhost:5001/

In [ ]:
# Edit source on the host — simulates a developer making a change
with open("/tmp/devapp/app.py", "w") as f:
    f.write(textwrap.dedent("""\
        from flask import Flask
        app = Flask(__name__)

        @app.route("/")
        def index():
            return "Version 2 — updated on host!\\n"

        if __name__ == '__main__':
            app.run(host='0.0.0.0', port=5000, debug=True)
    """))

# Flask debug mode detects the file change and restarts
time.sleep(3)
!curl -s http://localhost:5001/

!docker rm -f flask-dev

## 3. Read-Only Bind Mounts for Configuration

Configuration files (TLS certificates, app config, secrets) can be injected into a container at runtime via read-only bind mounts — without baking them into the image.

```bash
# Inject a config file — container can read but not modify it
docker run \
    --mount type=bind,source=/etc/myapp/config.yaml,target=/app/config.yaml,readonly \
    myapp

# Inject TLS certificates
docker run \
    --mount type=bind,source=/etc/letsencrypt/live/example.com,target=/certs,readonly \
    nginx:alpine
```

This is cleaner than baking config into the image because:
- The same image works in dev, staging, and production with different configs
- Secrets are not committed to the image or its history
- Config changes don't require a rebuild

In [ ]:
# Write a config file on the host
with open("/tmp/app-config.yaml", "w") as f:
    f.write("env: production\nlog_level: info\nmax_connections: 100\n")

# Mount as read-only single file
!docker run --rm \
    --mount type=bind,source=/tmp/app-config.yaml,target=/app/config.yaml,readonly \
    alpine sh -c '
        echo "Config contents:"
        cat /app/config.yaml
        echo ""
        echo "Trying to modify..."
        echo "hacked" >> /app/config.yaml 2>&1 || echo "Write blocked — read-only mount"
    '

!rm /tmp/app-config.yaml

## 4. Bind Mount Pitfalls

### UID/GID mismatch

Files created on the host belong to your user (e.g. UID 1000). Inside the container, the process may run as a different UID. This causes permission errors.

```bash
# Container runs as root (UID 0) by default — reads your files fine
# But if you use USER in the Dockerfile (UID 1001), it may not have read access
# to files owned by host UID 1000
```

Solutions:
- Run the container with `--user $(id -u):$(id -g)` to match host UID/GID
- Set directory permissions to be world-readable (`chmod 755`)
- Use volumes instead — Docker manages ownership

### macOS and Windows performance

On macOS and Windows, Docker runs inside a VM. Bind mounts cross the VM boundary — every file read/write goes through a translation layer. For large source trees with many small files (e.g. `node_modules`), this can be 10–100× slower than native.

Workarounds:
- Use **Docker Desktop's VirtioFS** (default on recent Docker Desktop) — significantly faster than the legacy osxfs
- Use a **named volume for `node_modules`** — keep dependencies inside the VM, only mount source:

```bash
docker run \
    -v $(pwd):/app \
    -v node_modules:/app/node_modules \
    node:20 npm start
# Source code bind-mounted (updated on save)
# node_modules in a volume (fast, inside the VM)
```

### Absolute paths required

Bind mount sources must be absolute paths. Use `$(pwd)` or `${PWD}` for the current directory:

```bash
docker run -v $(pwd):/app myimage    # correct
docker run -v ./src:/app/src myimage # WRONG — relative paths fail
```

In [ ]:
# UID/GID demo: check what UID/GID the container process runs as
print("Container default (root):")
!docker run --rm alpine id

import subprocess
host_uid = subprocess.check_output(["id", "-u"]).decode().strip()
host_gid = subprocess.check_output(["id", "-g"]).decode().strip()
print(f"\nContainer matching host UID {host_uid}:{host_gid}:")
!docker run --rm --user {host_uid}:{host_gid} alpine id

## 5. Mounting the Docker Socket

A special bind mount pattern: mounting the Docker daemon socket into a container gives that container full control over the Docker daemon — it can create, start, and stop containers on the host.

```bash
docker run -v /var/run/docker.sock:/var/run/docker.sock docker:cli docker ps
```

**Use cases:** CI agents (Jenkins, GitLab Runner) that need to build Docker images, monitoring tools (Portainer, ctop) that need daemon access.

**Security warning:** Mounting the Docker socket gives the container **root-equivalent access to the host**. Anyone who can run commands in that container can escape to the host. Only do this in trusted, controlled environments — never in production containers that handle untrusted input.

In [ ]:
# Container that can see other Docker containers via socket mount
!docker run --rm \
    -v /var/run/docker.sock:/var/run/docker.sock \
    docker:cli \
    docker ps --format 'table {{.Names}}\t{{.Image}}\t{{.Status}}' 2>/dev/null || \
    echo "Note: docker:cli image may need pulling — docker pull docker:cli"

## 6. tmpfs Mounts

A `tmpfs` mount creates an **in-memory filesystem** inside the container. Data written to it never touches the host disk and is lost when the container stops.

```bash
docker run --mount type=tmpfs,target=/tmp myapp
# or short form (Linux only):
docker run --tmpfs /tmp myapp
```

### Why use tmpfs?

| Use case | Why tmpfs |
|----------|----------|
| Sensitive data (tokens, keys) | Never written to disk — can't be recovered from disk forensics |
| Scratch space / temp files | High-speed writes; no disk I/O overhead |
| Cache directories | Fast, ephemeral — throw away on restart |
| `/run` and `/tmp` in hardened containers | No-exec tmpfs prevents writing executables |

### Options

```bash
--mount type=tmpfs,target=/tmp,tmpfs-size=64m   # limit to 64 MB RAM
--mount type=tmpfs,target=/tmp,tmpfs-mode=1777  # set permissions (octal)
```

In [ ]:
# tmpfs: fast writes, never persisted to disk
!docker run --rm \
    --mount type=tmpfs,target=/ramdisk,tmpfs-size=32m \
    alpine sh -c '
        echo "Writing to tmpfs (RAM)..."
        dd if=/dev/urandom of=/ramdisk/test.bin bs=1M count=10 2>&1 | tail -1

        echo "Mount info:"
        mount | grep ramdisk

        echo "File exists in container: $(ls -lh /ramdisk/test.bin)"
    '

In [ ]:
# Sensitive data in tmpfs: secrets loaded at runtime, never hit disk
!docker run --rm \
    --mount type=tmpfs,target=/run/secrets \
    -e SECRET_TOKEN=supersecretvalue123 \
    alpine sh -c '
        # Write the secret to tmpfs (RAM only)
        echo "$SECRET_TOKEN" > /run/secrets/token
        echo "Secret written to RAM:"
        cat /run/secrets/token
        echo "(This never touched the host disk)"
    '

In [ ]:
# tmpfs data does not survive container restart
!docker run -d --name tmpfs-demo \
    --mount type=tmpfs,target=/cache \
    alpine sleep 30

!docker exec tmpfs-demo sh -c 'echo "cached data" > /cache/data.txt && cat /cache/data.txt'

!docker restart tmpfs-demo

import time; time.sleep(1)
!docker exec tmpfs-demo sh -c 'ls /cache/ 2>&1 || echo "empty — tmpfs cleared on restart"'

!docker rm -f tmpfs-demo

## 7. Choosing the Right Storage Type

| Scenario | Best choice | Why |
|----------|-------------|-----|
| Database data (postgres, mysql) | **Volume** | Persistent, Docker-managed, no path coupling |
| Source code in development | **Bind mount** | Live reloading, edited from host |
| Config file injection | **Bind mount (read-only)** | Single file, no rebuild needed |
| Sharing data between containers | **Volume** | Named, easy to reference |
| Sensitive data (tokens, secrets) | **tmpfs** | Never hits disk |
| High-speed scratch / temp files | **tmpfs** | In-memory speed |
| CI build artifacts | **Volume** | Persist across container restarts in CI |
| Backing up data | **Volume** | Consistent, restorable |
| Production app data | **Volume** | Portable, driver-swappable |

A common production pattern combines all three:

```bash
docker run \
    -v pgdata:/var/lib/postgresql/data \
    --mount type=bind,source=/etc/myapp/pg.conf,target=/etc/postgresql/postgresql.conf,readonly \
    --mount type=tmpfs,target=/run/postgresql \
    postgres:16
# Volume:     database files — persistent
# Bind mount: custom config — injected from host, read-only
# tmpfs:      runtime socket and PID files — ephemeral, no disk I/O
```

## Summary

- **Bind mounts** map a specific host path into the container. Changes on the host are immediately visible inside the container and vice versa. Use them for development (live code reloading) and config injection.
- Use `--mount type=bind,source=ABSOLUTE_PATH,target=/path` — sources must be absolute paths.
- Add `readonly` to prevent the container from writing back to the host path.
- **Pitfalls:** UID/GID mismatch can cause permission errors — use `--user $(id -u):$(id -g)`. On macOS/Windows, bind mounts are slower due to the VM boundary — use VirtioFS or keep large dependency trees in named volumes.
- **Mounting the Docker socket** (`/var/run/docker.sock`) gives the container root-equivalent host access — only in trusted environments.
- **tmpfs** creates an in-memory filesystem. Data is fast, never written to disk, and lost when the container stops. Use for sensitive secrets, scratch space, and cache that should not persist.
- **Decision guide:** volumes for persistent data and sharing between containers; bind mounts for development and config injection; tmpfs for secrets and ephemeral scratch space.